# Beca 18 RAG Chatbot
Retrieval-Augmented Generation system for Peru's Beca 18 scholarship regulations.

**Source:** Resolución Directoral Ejecutiva N.° 033-2026-MINEDU/VMGI-PRONABEC

## Setup

In [ ]:
# Install dependencies (run in Colab)
# !pip install chromadb==0.6.3 google-generativeai==0.8.5 ipywidgets==8.1.5 pypdf==5.4.0 python-dotenv==1.1.0 tiktoken==0.9.0

In [ ]:
import os

try:
    # Google Colab: read from Secrets
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except ImportError:
    # Local: read from .env file
    from dotenv import load_dotenv
    load_dotenv()
    GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

print("API key loaded:", "✓" if GEMINI_API_KEY else "✗ NOT FOUND")

## Step 1: PDF Extraction
Extract text from the PDF preserving page markers so we can cite sources later.

In [ ]:
from pypdf import PdfReader

# Works in both Colab (uploaded to /content/) and locally (data/ folder)
if os.path.exists("/content/beca18_reglamento.pdf"):
    PDF_PATH = "/content/beca18_reglamento.pdf"
else:
    PDF_PATH = "../data/beca18_reglamento.pdf"

def extract_pages(pdf_path: str) -> list[dict]:
    """Return a list of {page, text} dicts for every page in the PDF."""
    reader = PdfReader(pdf_path)
    pages = []
    for i, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        text = text.strip()
        if text:
            pages.append({"page": i, "text": text})
    return pages

pages = extract_pages(PDF_PATH)
print(f"Pages extracted: {len(pages)}")
print(f"\n--- Page 1 preview ---\n{pages[0]['text'][:500]}")

## Step 2: Token Counting and Chunking

We use `tiktoken` (cl100k_base encoding) to count tokens and split text into chunks.

**Chunk size justification:** 400 tokens balances context richness (enough to answer most questions) and retrieval precision (small enough to stay on-topic). The 60-token overlap prevents answers from being split across chunk boundaries.

In [ ]:
import tiktoken

CHUNK_SIZE = 400
CHUNK_OVERLAP = 60

enc = tiktoken.get_encoding("cl100k_base")

def count_tokens(text: str) -> int:
    return len(enc.encode(text))

def chunk_page(page_num: int, text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[dict]:
    """Split a page's text into overlapping token-based chunks."""
    tokens = enc.encode(text)
    chunks = []
    start = 0
    while start < len(tokens):
        end = min(start + chunk_size, len(tokens))
        chunk_text = enc.decode(tokens[start:end])
        chunks.append({
            "page": page_num,
            "chunk_index": len(chunks),
            "text": chunk_text,
            "token_count": end - start,
        })
        if end == len(tokens):
            break
        start += chunk_size - overlap
    return chunks

def chunk_all_pages(pages: list[dict]) -> list[dict]:
    all_chunks = []
    for p in pages:
        all_chunks.extend(chunk_page(p["page"], p["text"]))
    return all_chunks

chunks = chunk_all_pages(pages)

total_tokens = sum(count_tokens(p["text"]) for p in pages)
print(f"Total tokens in document : {total_tokens}")
print(f"Total chunks produced    : {len(chunks)}")
print(f"Avg tokens per chunk     : {sum(c['token_count'] for c in chunks) / len(chunks):.1f}")
print(f"\n--- Chunk 0 preview ---\n{chunks[0]['text'][:300]}")

## Step 3: Embeddings
Dual embedding functions using `gemini-embedding-001` with rate-limit handling.

In [ ]:
import time
import google.generativeai as genai

genai.configure(api_key=GEMINI_API_KEY)

EMBEDDING_MODEL = "models/gemini-embedding-001"

def embed_query(text: str) -> list[float]:
    """Embed a single query string."""
    response = genai.embed_content(
        model=EMBEDDING_MODEL,
        content=text,
        task_type="retrieval_query",
    )
    return response["embedding"]

def embed_documents(texts: list[str], batch_size: int = 20) -> list[list[float]]:
    """Embed a list of document chunks in batches with rate-limit backoff."""
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        for attempt in range(5):
            try:
                response = genai.embed_content(
                    model=EMBEDDING_MODEL,
                    content=batch,
                    task_type="retrieval_document",
                )
                embeddings.extend(response["embedding"])
                break
            except Exception as e:
                if attempt == 4:
                    raise
                wait = 2 ** attempt
                print(f"Rate limit hit, retrying in {wait}s... ({e})")
                time.sleep(wait)
        time.sleep(0.5)  # stay within quota
    return embeddings

# Quick sanity check
test_embedding = embed_query("¿Qué es Beca 18?")
print(f"Embedding dimension: {len(test_embedding)}")

## Step 4: ChromaDB — Persistent Vector Store
Idempotent indexing: skips embedding if the collection already has the same number of chunks.

In [ ]:
import chromadb

CHROMA_PATH = "../chroma_db"
COLLECTION_NAME = "beca18"

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma_client.get_or_create_collection(name=COLLECTION_NAME)

def index_chunks(chunks: list[dict]) -> None:
    if collection.count() == len(chunks):
        print(f"Collection already has {len(chunks)} chunks — skipping indexing.")
        return

    print(f"Embedding {len(chunks)} chunks...")
    texts = [c["text"] for c in chunks]
    embeddings = embed_documents(texts)

    collection.upsert(
        ids=[f"chunk_{i}" for i in range(len(chunks))],
        embeddings=embeddings,
        documents=texts,
        metadatas=[{"page": c["page"], "chunk_index": c["chunk_index"]} for c in chunks],
    )
    print(f"Indexed {len(chunks)} chunks into ChromaDB.")

index_chunks(chunks)
print(f"Total documents in collection: {collection.count()}")

## Step 5: Semantic Search
Returns the top-k most relevant chunks with text, metadata, and distance scores.

In [ ]:
def semantic_search(query: str, k: int = 5) -> list[dict]:
    """Return top-k chunks most relevant to the query."""
    query_embedding = embed_query(query)
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k,
        include=["documents", "metadatas", "distances"],
    )
    hits = []
    for text, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
        hits.append({"text": text, "page": meta["page"], "distance": dist})
    return hits

# Quick test
sample_hits = semantic_search("¿Cuáles son los requisitos de elegibilidad?", k=3)
for h in sample_hits:
    print(f"Page {h['page']} | distance {h['distance']:.4f}\n{h['text'][:200]}\n")

## Step 6: Grounded Generation
The LLM may only answer from the retrieved context. If the answer is not in the context it must say so. Every claim is backed by a page citation.

In [ ]:
GENERATION_MODEL = "gemini-1.5-flash"

SYSTEM_PROMPT = """Eres un asistente experto en el reglamento de Beca 18 (Resolución Directoral Ejecutiva N.° 033-2026-MINEDU/VMGI-PRONABEC).
Responde ÚNICAMENTE basándote en los fragmentos del documento proporcionados.
Al final de cada afirmación, indica entre paréntesis la página de donde proviene, por ejemplo: (Página 5).
Si la respuesta no se encuentra en los fragmentos, responde exactamente: "No encuentro esa información en el reglamento de Beca 18."
No inventes ni supongas información que no esté en el contexto."""

def build_context(hits: list[dict]) -> str:
    parts = []
    for h in hits:
        parts.append(f"[Página {h['page']}]\n{h['text']}")
    return "\n\n".join(parts)

def generate_answer(question: str, k: int = 5) -> dict:
    """Retrieve relevant chunks and generate a grounded answer."""
    hits = semantic_search(question, k=k)
    context = build_context(hits)
    prompt = f"{SYSTEM_PROMPT}\n\nCONTEXTO:\n{context}\n\nPREGUNTA: {question}\n\nRESPUESTA:"

    model = genai.GenerativeModel(GENERATION_MODEL)
    response = model.generate_content(prompt)
    return {"answer": response.text, "sources": hits}

# Quick test
result = generate_answer("¿Cuáles son los requisitos de elegibilidad para Beca 18?")
print(result["answer"])

## Step 7: Chat Interface
Interactive widget with question input, Ask/Clear buttons, k-value slider, and collapsible source fragments.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML

# --- UI components ---
question_input = widgets.Textarea(
    placeholder="Escribe tu pregunta sobre Beca 18...",
    layout=widgets.Layout(width="100%", height="80px"),
)

k_slider = widgets.IntSlider(
    value=5, min=1, max=10, step=1,
    description="Fuentes (k):",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="400px"),
)

ask_button = widgets.Button(
    description="Preguntar",
    button_style="primary",
    icon="search",
)

clear_button = widgets.Button(
    description="Limpiar",
    button_style="warning",
    icon="trash",
)

answer_output = widgets.Output()
sources_accordion = widgets.Accordion(children=[], titles=[])

# --- Handlers ---
def on_ask(b):
    question = question_input.value.strip()
    if not question:
        return

    ask_button.disabled = True
    ask_button.description = "Buscando..."
    answer_output.clear_output()

    with answer_output:
        try:
            result = generate_answer(question, k=k_slider.value)
            display(HTML(f"<b>Respuesta:</b><br>{result['answer'].replace(chr(10), '<br>')}"))

            # Build collapsible source panels
            panels = []
            titles = []
            for i, hit in enumerate(result["sources"]):
                out = widgets.Output()
                with out:
                    display(HTML(
                        f"<small><b>Distancia:</b> {hit['distance']:.4f}</small>"
                        f"<pre style='white-space:pre-wrap'>{hit['text']}</pre>"
                    ))
                panels.append(out)
                titles.append(f"Fragmento {i+1} — Página {hit['page']}")

            sources_accordion.children = panels
            for i, title in enumerate(titles):
                sources_accordion.set_title(i, title)

        except Exception as e:
            display(HTML(f"<span style='color:red'>Error: {e}</span>"))

    ask_button.disabled = False
    ask_button.description = "Preguntar"

def on_clear(b):
    question_input.value = ""
    answer_output.clear_output()
    sources_accordion.children = []

ask_button.on_click(on_ask)
clear_button.on_click(on_clear)

# --- Layout ---
buttons = widgets.HBox([ask_button, clear_button])
ui = widgets.VBox([
    widgets.HTML("<h3>Chatbot Reglamento Beca 18</h3>"),
    question_input,
    k_slider,
    buttons,
    answer_output,
    widgets.HTML("<b>Fragmentos recuperados:</b>"),
    sources_accordion,
])

display(ui)

## Testing
### On-topic questions
1. ¿Cuáles son los requisitos de elegibilidad para Beca 18?
2. ¿Cuáles son las modalidades de la beca?
3. ¿Cuáles son los montos de los estipendios?
4. ¿Cuáles son las obligaciones del becario?
5. ¿En qué casos se pierde la beca?

### Off-topic question
6. ¿Cuál es la capital de Francia?